In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph,END
from langgraph.graph.message import add_messages
from langchain.tools import tool
from langgraph.prebuilt import ToolNode
from langchain_core.messages import AIMessage,HumanMessage,SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict,Annotated

In [2]:
load_dotenv()

True

In [3]:
class MyState(TypedDict):
    receiver:str
    draft:str
    approval:str
    review_comments : str
    messages: Annotated[list,add_messages]

In [4]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [5]:
@tool
def send_email(draft:str)->dict:
    """This tool sends email"""
    print(draft)
    print("Email sent!")



In [6]:
llm_with_tools  = llm.bind_tools([send_email])

In [7]:
def llm_node(state:MyState)->dict:
    system = SystemMessage(content="You are a Helpful assitent.")
    response = llm_with_tools.invoke([system]+state["messages"])
    return {'messages':[response]}

def generate_email_draft(state:MyState)->dict:
    mes = state['messages']
    print("This is state:",state)
    if state.get('approval') is None:
        data = HumanMessage(content=f"reciver name is {state['receiver']}")
        draft = llm_with_tools.invoke(mes+[data])
    else:
        comments = HumanMessage(content=state['review_comments'])
        draft = llm_with_tools.invoke([comments]+mes)
    return {'draft':draft.content}


send_email_node = ToolNode([send_email])

def review_email(state: MyState) -> dict:
    """
    This node now acts as a 'Gate'. 
    It doesn't ask for input; it just reads the input 
    that was injected while the graph was paused.
    """
    print(f"--- [Node: Review] Checking approval status: {state['approval']} ---")
    
    if state['approval'] == 'edit':
        print(f"--- Reviewer requested changes: {state['review_comments']} ---")
        
    return {} # We don't need to return much, as the 'approval' is already in the state
        



In [8]:
# define routing function
def after_review(state:MyState):
    last = state["messages"][-1]

    if getattr(last,"tool_calls") and state['approval'] == 'yes':
        return "send_email"
    elif state['approval'] == 'edit':
        return "draft"
    return "end"


In [9]:
flow = StateGraph(MyState)
flow.add_node("llm_node",llm_node)
flow.add_node("draft",generate_email_draft)
flow.add_node("review",review_email)
flow.add_node("send",send_email_node)

flow.set_entry_point("llm_node")

In [11]:
flow.add_edge("llm_node","draft")
flow.add_edge("draft","review")
flow.add_edge("review","send")
flow.add_conditional_edges("review",
                           after_review,
                           {
                               'send_email':'send',
                               'draft':'draft',
                               'end':END
                           })                           

In [12]:
memory = MemorySaver()

In [13]:
app = flow.compile(checkpointer=memory,interrupt_after=["draft"])
config1 = {'configurable':{'thread_id':'Narsimha_thread'}}
config2 = {'configurable':{'thread_id':'Anand_thread'}}

In [14]:
result = app.invoke({'messages':[HumanMessage(content="write a very short email to my friend.")],'receiver':'Narsimha'},
                    config = config1)

This is state: {'receiver': 'Narsimha', 'messages': [HumanMessage(content='write a very short email to my friend.', additional_kwargs={}, response_metadata={}, id='06359f76-2b36-49e6-84a9-0e3ffff59382'), AIMessage(content='Subject: Quick Hello  \n\nHey [Friend’s Name],\n\nJust wanted to say hi and see how you’re doing. Let’s catch up soon!\n\nBest,  \n[Your Name]', additional_kwargs={'reasoning_content': "The user wants a very short email to their friend. We need to output the email text. No need to send email unless they ask. Just write email. Provide maybe subject and body. We'll just give a short email."}, response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 137, 'total_tokens': 228, 'completion_time': 0.198027092, 'completion_tokens_details': {'reasoning_tokens': 46}, 'prompt_time': 0.005610369, 'prompt_tokens_details': None, 'queue_time': 0.275201604, 'total_time': 0.203637461}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_f16b1963fe', 'se

In [15]:
print(result)

{'receiver': 'Narsimha', 'draft': 'Subject: Quick Hello  \n\nHey Narsimha,\n\nJust wanted to say hi and see how you’re doing. Let’s catch up soon!\n\nBest,  \n[Your Name]', 'messages': [HumanMessage(content='write a very short email to my friend.', additional_kwargs={}, response_metadata={}, id='06359f76-2b36-49e6-84a9-0e3ffff59382'), AIMessage(content='Subject: Quick Hello  \n\nHey [Friend’s Name],\n\nJust wanted to say hi and see how you’re doing. Let’s catch up soon!\n\nBest,  \n[Your Name]', additional_kwargs={'reasoning_content': "The user wants a very short email to their friend. We need to output the email text. No need to send email unless they ask. Just write email. Provide maybe subject and body. We'll just give a short email."}, response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 137, 'total_tokens': 228, 'completion_time': 0.198027092, 'completion_tokens_details': {'reasoning_tokens': 46}, 'prompt_time': 0.005610369, 'prompt_tokens_details': None, '

In [16]:
snapshot = app.get_state(config1)
print(snapshot)
# User need to review content from state now

StateSnapshot(values={'receiver': 'Narsimha', 'draft': 'Subject: Quick Hello  \n\nHey Narsimha,\n\nJust wanted to say hi and see how you’re doing. Let’s catch up soon!\n\nBest,  \n[Your Name]', 'messages': [HumanMessage(content='write a very short email to my friend.', additional_kwargs={}, response_metadata={}, id='06359f76-2b36-49e6-84a9-0e3ffff59382'), AIMessage(content='Subject: Quick Hello  \n\nHey [Friend’s Name],\n\nJust wanted to say hi and see how you’re doing. Let’s catch up soon!\n\nBest,  \n[Your Name]', additional_kwargs={'reasoning_content': "The user wants a very short email to their friend. We need to output the email text. No need to send email unless they ask. Just write email. Provide maybe subject and body. We'll just give a short email."}, response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 137, 'total_tokens': 228, 'completion_time': 0.198027092, 'completion_tokens_details': {'reasoning_tokens': 46}, 'prompt_time': 0.005610369, 'prompt_tok

In [21]:
#call next nodes
snapshot = app.update_state(config1,{
    'approval':'edit',
    'review_comments':'ask him to send his current address.'
}
)

In [22]:
res = app.invoke(None,config1)
print(res)

--- [Node: Review] Checking approval status: edit ---
--- Reviewer requested changes: ask him to send his current address. ---
This is state: {'receiver': 'Narsimha', 'draft': 'Subject: Quick Hello  \n\nHey Narsimha,\n\nJust wanted to say hi and see how you’re doing. Let’s catch up soon!\n\nBest,  \n[Your Name]', 'approval': 'edit', 'review_comments': 'ask him to send his current address.', 'messages': [HumanMessage(content='write a very short email to my friend.', additional_kwargs={}, response_metadata={}, id='06359f76-2b36-49e6-84a9-0e3ffff59382'), AIMessage(content='Subject: Quick Hello  \n\nHey [Friend’s Name],\n\nJust wanted to say hi and see how you’re doing. Let’s catch up soon!\n\nBest,  \n[Your Name]', additional_kwargs={'reasoning_content': "The user wants a very short email to their friend. We need to output the email text. No need to send email unless they ask. Just write email. Provide maybe subject and body. We'll just give a short email."}, response_metadata={'token_usa

Time Travel